# 03 · Data and scoring (stage S1, GQA + TextVQA) and the evaluation runner

This notebook checks the data side of stage S1 and the evaluation runner.
- **GQA splits:** test-dev is cut into test / ctrl_train / dev / lora_train. No image may sit in two splits, so dev scores never come from an image seen elsewhere.
- **GQA items, extraction, scoring:** dev items load as `Item`, and hand-written model outputs give the expected answer and score.
- **TextVQA:** image-disjoint splits, items with 10 answers, and VQA accuracy with the lmms-eval answer processing. No model is needed for these.
- **Evaluation runner:** `N_ITEMS` GQA dev items at `R=3` with full tokens. Rows carry provenance, and re-running resumes without new work.
- **Audit:** predictions in saved result files that the current scorers mark wrong, next to the references.

Thinking is still on here (stage S2 turns it off), so the parse rate is shown, not asserted.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import socket
from collections import Counter

In [ ]:
from adaloop import api  # importing api first; setup_notebook puts the repo root on sys.path

GPU_GROUP = None  # None = first free pair in gpu_groups; 0 or 1 forces a pair; changing it needs a kernel restart
ROOT, cfg = api.setup_notebook("configs/mvp.yaml", gpu_group=GPU_GROUP)
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
# torch is imported here because the GPU list needs it; the other third-party imports come after this cell
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("chosen GPUs, physical ids CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"chosen GPU {i} (torch index) | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")
else:
    print("no CUDA GPU visible: the model steps will fail, the rest of the notebook can still be read")

In [ ]:
import pandas as pd

In [ ]:
# Settings: batch size and loop count for the GQA runs here
N_ITEMS = 20  # dev items per benchmark in steps 2, 4 and 5
R = 3  # loop passes for every evaluation here

### Step-1:- GQA splits

GQA test-dev is cut into test, ctrl_train, dev and lora_train with the sizes in `configs/mvp.yaml`. The splits are question-disjoint and image-disjoint, so a dev score never comes from an image the other splits have seen. The manifest hash must stay the same on rebuild.

In [ ]:
manifest = api.build_gqa_splits(cfg, ROOT)
sizes = {name: len(ids) for name, ids in manifest["splits"].items()}
n_images = {name: len(ids) for name, ids in manifest["images"].items()}
pd.DataFrame({"questions": sizes, "images": n_images})

#### Split sizes

In [ ]:
for name, want in cfg["splits"]["sizes"].items():
    if want is not None:
        assert sizes[name] == want, f"GQA split {name}: {sizes[name]} vs {want}"
print("✓ GQA split sizes match configs/mvp.yaml")

#### Disjointness and manifest hash

In [ ]:
names = list(manifest["splits"])
pairs = [(a, b) for i, a in enumerate(names) for b in names[i + 1:]]
for a, b in pairs:
    assert not set(manifest["splits"][a]) & set(manifest["splits"][b]), f"questions shared by {a} and {b}"
    assert not set(manifest["images"][a]) & set(manifest["images"][b]), f"images shared by {a} and {b}"
stored = api.load_manifest(ROOT / cfg["benchmarks"]["gqa"]["manifest"])["hash"]
assert stored == manifest["hash"], f"manifest hash changed on rebuild: {stored} vs {manifest['hash']}"
print("✓ GQA splits: question-disjoint, image-disjoint, manifest hash stable")

### Step-2:- GQA items

A dev item is an `Item` with the image, the question, the answers and the question type. The first few are printed so the fields can be read by eye, and the image of the first item is shown.

In [ ]:
items = api.load_gqa(cfg, "dev", limit=N_ITEMS, root=ROOT)
for it in items[:5]:
    print(f"{it.item_id:>10} {it.question_type:8} {it.image.size} {it.question!r} -> {it.answers}")
print("question types", Counter(it.question_type for it in items))

In [ ]:
items[0].image

In [ ]:
n_item_images = len({it.image_id for it in items})
print("distinct image_id", n_item_images)
assert len(items) == N_ITEMS, f"got {len(items)} items, want {N_ITEMS}"
assert all(it.answers and it.answers[0] for it in items), "an item has no answer"
assert all(it.split == "dev" and it.item_id in manifest["splits"]["dev"] for it in items), "an item is not a dev item in the manifest"
assert n_item_images > 1, f"items cover only {n_item_images} image"
print("✓ GQA items: count, answers, dev split, several images")

### Step-3:- Answer extraction and scoring

The model output is free text. `extract_answer` cuts out the answer after the thinking and end markers, and returns `None` when the output was cut before the answer (`hit_limit`). `score_gqa` is the lmms-eval GQA exact match: case and ASCII punctuation are ignored, while articles and spaces still count. The cases below are hand-written outputs with known answers.

In [ ]:
cases = [  # (raw output, hit_limit, expected answer)
    ("<|content_text|>Yes<|end_message|>", False, "Yes"),
    ("<|content_thinking|>the sky is blue<|content_text|>Answer: blue.<|end_message|>", False, "blue."),
    ("<|content_thinking|>still thinking about it", True, None),
    ("<|content_text|>a long answer that got cut", True, None),
    ("left", False, "left"),
]
for raw, hit, want in cases:
    got = api.extract_answer(raw, hit)
    print(f"{raw[:60]!r:64} -> {got!r}")
    assert got == want, f"extract_answer({raw[:60]!r}, hit_limit={hit}) gave {got!r}, want {want!r}"
print("✓ extraction on hand-written outputs")

In [ ]:
assert api.score_gqa("Blue.", ["blue"]) == 1.0, "exact match after normalization failed"
assert api.score_gqa("dark blue", ["blue"]) == 0.0, "a mismatch scored above 0"
assert api.score_gqa(None, ["blue"]) == 0.0, "an unparsed answer scored above 0"
assert api.score_gqa("t-shirt", ["tshirt"]) == 1.0, "the hyphen was not removed"
assert api.score_gqa("Yes.", ["yes"]) == 1.0, "case or the final period was not ignored"
assert api.score_gqa("the countertop", ["countertop"]) == 0.0, "an article was stripped; lmms-eval keeps it"
print("✓ GQA scoring: exact match, mismatch, unparsed, punctuation")

### Step-4:- TextVQA: splits, items, VQA accuracy

TextVQA is the second benchmark, scored by VQA accuracy over 10 human answers. It gets the same image-disjoint split rule as GQA. No model runs in this step.

In [ ]:
tv_manifest = api.build_textvqa_splits(cfg, ROOT)
tv_sizes = {name: len(ids) for name, ids in tv_manifest["splits"].items()}
tv_images = {name: set(ids) for name, ids in tv_manifest["images"].items()}
pd.DataFrame({"questions": tv_sizes, "images": {name: len(ids) for name, ids in tv_images.items()}})

In [ ]:
for name, want in cfg["splits"]["sizes"].items():
    if want is not None:
        assert tv_sizes[name] == want, f"TextVQA split {name}: {tv_sizes[name]} vs {want}"
assert sum(map(len, tv_images.values())) == len(set().union(*tv_images.values())), "an image is in two TextVQA splits"
print("✓ TextVQA splits: sizes match, image-disjoint")

In [ ]:
tv_items = api.load_textvqa(cfg, "dev", limit=N_ITEMS, root=ROOT)
for it in tv_items[:3]:
    print(f"{it.item_id:>6} {it.image.size} {it.question!r}\n       {it.answers}")

In [ ]:
tv_items[0].image

In [ ]:
assert len(tv_items) == N_ITEMS, f"got {len(tv_items)} TextVQA items, want {N_ITEMS}"
assert all(len(it.answers) == 10 for it in tv_items), "a TextVQA item does not have 10 answers"
print("✓ TextVQA items: count and 10 answers each")

In [ ]:
cases = [  # (name, prediction, 10 reference answers, expected score)
    ("all match", "stop", ["stop"] * 10, 1.0),
    ("1 of 10 match", "stop", ["stop"] + ["go"] * 9, 0.3),
    ("no match", "left", ["stop"] * 10, 0.0),
    ("article + punctuation", "The Coca-Cola.", ["coca cola"] * 10, 1.0),
    ("unparsed", None, ["stop"] * 10, 0.0),
]
for name, pred, answers, want in cases:
    got = api.score_textvqa(pred, answers)
    assert abs(got - want) < 1e-9, f"TextVQA {name}: {pred!r} -> {got}, want {want}"
print("✓ TextVQA scoring: all cases")

### Step-5:- Evaluation runner (R=3, full tokens)

`run_evaluation` runs the model on each item at `R` loop passes and writes one row per item to a `.jsonl` file under `execution_results/`. Each row carries the config hash, git hash and GPU, so a result can be traced back. The file name carries the config hash too. Running again with `resume=True` reads that file and adds no new rows.

#### Load the model

In [ ]:
print("[STAGE] load model")
vlm = api.load_model(cfg)

#### Batch run

In [ ]:
print("[STAGE] evaluation")
out = ROOT / cfg["out_dir"] / "nb03" / f"gqa_dev_n{N_ITEMS}_res{cfg['res_px']}_R{R}_{api.config_hash(cfg)}.jsonl"
summary = api.run_evaluation(vlm, items, cfg, R, out, resume=True)
print("output", out)
summary

#### Rows and provenance

In [ ]:
rows = api.read_results(out)
df = pd.DataFrame(rows)
print("raw output example:", repr(rows[0]["raw_output"][:300]))
print(f"accuracy {summary['accuracy']:.1f}  parse rate {summary['parse_rate']:.1f}  (thinking still on; S2 fixes parsing)")
df[["item_id", "question_type", "pred", "answers", "correct", "parse_ok", "n_vis", "n_out_tokens", "generate_s"]].head(10)

In [ ]:
assert len(rows) == N_ITEMS and df.item_id.nunique() == N_ITEMS, f"{len(rows)} rows for {N_ITEMS} items"
assert all(r.get(k) for r in rows for k in ("config_hash", "git_hash", "gpu_type")), "a row is missing provenance"
assert (df.k == 1.0).all() and (df.R == R).all(), "an item did not run at full tokens and R"
print("✓ evaluation rows: one per item, provenance, full tokens at R")

#### Resume

In [ ]:
print("[STAGE] resume")
again = api.run_evaluation(vlm, items, cfg, R, out, resume=True)
assert len(api.read_results(out)) == N_ITEMS, "resume added rows"
assert again["accuracy"] == summary["accuracy"], "resume changed the accuracy"
print("✓ resume: no new rows, same accuracy")

### Step-6:- Audit of wrong predictions

Reading the misses shows whether the model or the verifier is wrong. Saved rows are scored again from `pred` and `answers` with the current scorers (nb04 variant-B files if present, else any `.jsonl` under `execution_results/`); "wrong" means a score of 0. The files may come from an older prompt, so long or wordy predictions need not repeat with the current one. No model is needed.

In [ ]:
N_SHOW = 20  # wrong predictions printed per benchmark
files, wrong = api.wrong_predictions(ROOT / "execution_results")
print(f"{len(files)} result files read | {len(wrong)} wrong predictions")
for f in files:
    print(" ", f.relative_to(ROOT))

In [ ]:
for benchmark in cfg["benchmarks"]:
    misses = [r for r in wrong if r["benchmark"] == benchmark]
    print(f"\n{benchmark}: {len(misses)} wrong, showing {min(N_SHOW, len(misses))}")
    for r in misses[:N_SHOW]:
        print(f"  {r['item_id']:>10} pred {r['pred']!r}  |  refs {dict(Counter(r['answers']))}")

In [ ]:
print("✓ notebook 03 done")